# Ray Cluster Autoscaling on Red Hat OpenShift AI

This notebook demonstrates the full lifecycle of a Ray cluster with **in-tree autoscaling**
on RHOAI: create the cluster, submit a bursty CPU workload, observe workers scale up and
back down, then tear the cluster down.

It uses the CodeFlare SDK (`enable_autoscaling=True`, `min_workers`, `max_workers`) and
the companion script `autoscaling_load.py`.

> **Kueue:** Autoscaling is not supported when Kueue manages RayClusters in your namespace.
Use a non-Kueue project or see [RHAIRFE-909](https://redhat.atlassian.net/browse/RHAIRFE-909).

### Prerequisites

- **Red Hat OpenShift AI 3.5+** with KubeRay and CodeFlare SDK autoscaling support
- A **non-Kueue** data science project namespace
- Enough cluster capacity for the head pod plus up to `max_workers` worker pods


## Setup

Run the **install** cell below if your workbench image does not include CodeFlare SDK 0.37+
(with `enable_autoscaling` support). Ray is installed automatically as a dependency of the SDK.

In [ ]:
%pip install -q "codeflare-sdk>=0.37.0,<1.0"

In [ ]:
import time

from codeflare_sdk import Cluster, ClusterConfiguration, TokenAuthentication
from codeflare_sdk.common.kubernetes_cluster import get_api_client
from kubernetes import client as k8s_client

## Step 1 — Authenticate

Set your OpenShift token and API server URL from `oc whoami -t` and `oc whoami --show-server`.


In [ ]:
auth = TokenAuthentication(
    token="",
    server="",
    skip_tls=False,
)
auth.login()

## Step 2 — Configure autoscaling cluster

The SDK sets `enableInTreeAutoscaling` on the RayCluster and maps `min_workers` /
`max_workers` to worker replica bounds.

NOTE: If running outside of RHOAI workbench pods, set `namespace="rhods-notebooks"`
(or your project namespace) in `ClusterConfiguration`.


In [ ]:
CLUSTER_NAME = "ray-autoscale"
MIN_WORKERS = 1
MAX_WORKERS = 2
LOAD_TASKS = 3
LOAD_SLEEP_S = 180

cluster = Cluster(
    ClusterConfiguration(
        name=CLUSTER_NAME,
        enable_autoscaling=True,
        min_workers=MIN_WORKERS,
        max_workers=MAX_WORKERS,
        head_cpu_requests=1,
        head_cpu_limits=1,
        head_memory_requests=7,
        head_memory_limits=8,
        worker_cpu_requests=1,
        worker_cpu_limits=1,
        worker_memory_requests=5,
        worker_memory_limits=6,
        head_extended_resource_requests={"nvidia.com/gpu": 0},
        worker_extended_resource_requests={"nvidia.com/gpu": 0},
    )
)

## Step 3 — Create the cluster


In [ ]:
cluster.apply()

In [ ]:
cluster.wait_ready()

In [ ]:
cluster.details()

## Step 4 — Verify initial worker count

Poll worker pods until `min_workers` is running. Use the `oc` command to inspect
RayCluster replica fields while the test runs.


In [ ]:
def count_workers(cluster_name: str, namespace: str) -> int:
    api = k8s_client.CoreV1Api(get_api_client())
    label = f"ray.io/node-type=worker,ray.io/cluster={cluster_name}"
    pods = api.list_namespaced_pod(namespace, label_selector=label)
    return len(pods.items or [])


def wait_for_worker_count(
    cluster_name: str,
    namespace: str,
    predicate,
    timeout_s: int = 900,
    poll_s: int = 10,
) -> int:
    start = time.time()
    last = None
    while time.time() - start < timeout_s:
        last = count_workers(cluster_name, namespace)
        print(f"Worker count: {last}")
        if predicate(last):
            return last
        time.sleep(poll_s)
    raise TimeoutError(
        f"Timed out waiting for worker count. cluster={cluster_name} last={last}"
    )


NAMESPACE = cluster.config.namespace
print(f"Namespace: {NAMESPACE}")
!oc get raycluster {CLUSTER_NAME} -n {NAMESPACE} -o yaml | grep -E 'enableInTreeAutoscaling|minReplicas|maxReplicas|replicas' || true

initial_workers = wait_for_worker_count(
    CLUSTER_NAME, NAMESPACE, lambda n: n == MIN_WORKERS, timeout_s=600
)
print(f"Initial worker count matches min_workers={MIN_WORKERS}: {initial_workers}")

## Step 5 — Submit autoscaling load job

Three single-CPU tasks should exceed head + one worker capacity and trigger scale-up.


In [ ]:
client = cluster.job_client

submission_id = client.submit_job(
    entrypoint=(
        f"AUTOSCALING_TASKS={LOAD_TASKS} AUTOSCALING_TASK_SLEEP_S={LOAD_SLEEP_S} "
        "python autoscaling_load.py"
    ),
    runtime_env={"working_dir": "./"},
)
print(f"Submitted autoscaling load job: {submission_id}")
print(f"Job status: {client.get_job_status(submission_id)}")

## Step 6 — Observe scale-up


In [ ]:
scaled_up = wait_for_worker_count(
    CLUSTER_NAME,
    NAMESPACE,
    lambda n: n >= MAX_WORKERS,
    timeout_s=900,
)
print(f"Scale-up observed: {scaled_up} worker(s) (max_workers={MAX_WORKERS})")
!oc get pods -n {NAMESPACE} -l ray.io/cluster={CLUSTER_NAME},ray.io/node-type=worker

## Step 7 — Wait for job completion and scale-down

Scale-down can take several minutes after the cluster becomes idle.


In [ ]:
while client.get_job_status(submission_id) in {"PENDING", "RUNNING"}:
    print(f"Job status: {client.get_job_status(submission_id)}")
    time.sleep(15)

print(f"Final job status: {client.get_job_status(submission_id)}")
print(client.get_job_logs(submission_id))

scaled_down = wait_for_worker_count(
    CLUSTER_NAME,
    NAMESPACE,
    lambda n: n == MIN_WORKERS,
    timeout_s=900,
)
print(f"Scale-down observed: {scaled_down} worker(s) (min_workers={MIN_WORKERS})")

## Step 8 — Cleanup


In [ ]:
client.delete_job(submission_id)
cluster.down()